In [2]:
!pip install fastapi uvicorn nest-asyncio trafilatura transformers torch requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 21.9 MB/s eta 0:00:00


In [3]:
!pip install transformers torch trafilatura

In [4]:
import trafilatura
from transformers import pipeline
import textwrap
import requests

class FakeNewsEngine:
    def __init__(self):
        print("🤖 Initializing Fake News Transformer Engine...")
        self.classifier = pipeline("text-classification", model="winterForestStump/Roberta-fake-news-detector")
        print("✅ Model loaded successfully!\n")

    def scrape_text(self, url):
        """Extracts text using a disguised browser request."""
        print(f"🌐 Scraping article from: {url}")

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
        }

        try:
            # 1. Fetch the HTML pretending to be a real human browser
            response = requests.get(url, headers=headers, timeout=10)

            # 2. Check if the site still blocked us
            if response.status_code != 200:
                print(f"⚠️ Warning: Site returned status code {response.status_code}")
                return None

            # 3. Extract the clean text from the downloaded HTML
            text = trafilatura.extract(response.text)
            return text

        except Exception as e:
            print(f"⚠️ Request failed: {e}")
            return None

    def analyze_article(self, url):
        """Scrapes the URL and runs the text through the Transformer model."""
        article_text = self.scrape_text(url)

        if not article_text:
            return {"error": "Could not extract text. The site has strict anti-bot firewalls."}

        # Transformers have a token limit. We grab the first 2000 chars.
        truncated_text = article_text[:2000]

        print("🧠 Analyzing linguistic patterns and semantics...")
        result = self.classifier(truncated_text)[0]

        label = result['label']
        confidence = round(result['score'] * 100, 2)

        return {
            "status": "success",
            "prediction": label,
            "confidence": confidence,
            "preview_text": textwrap.shorten(article_text, width=150, placeholder="...")
        }

if __name__ == "__main__":
    engine = FakeNewsEngine()
    test_urls = ["https://www.theonion.com/man-just-buying-one-of-every-cleaning-product-in-case-1819576081",
        "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)"]

    for url in test_urls:
        print("-" * 60)
        report = engine.analyze_article(url)

        if "error" in report:
            print(f"❌ Error: {report['error']}")
        else:
            print("\n📊 --- ANALYSIS REPORT ---")
            print(f"Snippet: {report['preview_text']}")

            if report['prediction'] == 'FAKE':
                print(f"Verdict: 🔴 FAKE/UNRELIABLE ({report['confidence']}% confidence)")
            else:
                print(f"Verdict: 🟢 REAL NEWS ({report['confidence']}% confidence)")
        print("-" * 60 + "\n")

🤖 Initializing Fake News Transformer Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/859 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

✅ Model loaded successfully!

------------------------------------------------------------
🌐 Scraping article from: https://www.theonion.com/man-just-buying-one-of-every-cleaning-product-in-case-1819576081
⚠️ Warning: Site returned status code 404
❌ Error: Could not extract text. The site has strict anti-bot firewalls.
------------------------------------------------------------

------------------------------------------------------------
🌐 Scraping article from: https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)
🧠 Analyzing linguistic patterns and semantics...

📊 --- ANALYSIS REPORT ---
Snippet: Transformer (deep learning) | This article has multiple issues. Please help improve it or discuss these issues on the talk page. (Learn how and...
Verdict: 🟢 REAL NEWS (100.0% confidence)
------------------------------------------------------------



In [5]:
import trafilatura
from transformers import pipeline
import textwrap
import requests
import re # Regular expressions for cleaning HTML

class FakeNewsEngine:
    def __init__(self):
        print("🤖 Initializing Fake News Transformer Engine...")
        self.classifier = pipeline("text-classification", model="winterForestStump/Roberta-fake-news-detector")
        print("✅ Model loaded successfully!\n")

    def scrape_text(self, url):
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
        }
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                return None
            return trafilatura.extract(response.text)
        except Exception:
            return None

    def fact_check_claim(self, keyword_query):
        """Searches Wikipedia API for debunked claims, using a browser disguise."""
        print(f"🔎 Cross-referencing databases for: '{keyword_query}'...")

        # We must use the disguise here too, otherwise Wikipedia blocks the API call!
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"
        }

        try:
            wiki_url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={keyword_query} debunked OR hoax OR conspiracy&utf8=&format=json"
            response = requests.get(wiki_url, headers=headers, timeout=10)

            # Make sure we got a successful 200 OK response
            if response.status_code == 200:
                wiki_res = response.json()

                if wiki_res['query']['search']:
                    evidence = []
                    for res in wiki_res['query']['search'][:2]: # Get top 2 results
                        # Clean the Wikipedia HTML snippet using regex
                        clean_snippet = re.sub(r'<[^>]+>', '', res['snippet'])
                        evidence.append(f"- Source: {res['title']} | Context: {clean_snippet[:120]}...")
                    return "\n".join(evidence)
        except Exception as e:
            return f"Fact-check search failed: {e}"

        return "No fact-check records found. Proceed with caution."

    def analyze_article(self, url, search_claim):
        """Runs the AI model AND the Fact Checker."""
        article_text = self.scrape_text(url)

        if not article_text:
            return {"error": "Could not extract text. Site might be blocking scrapers or link is dead."}

        # 1. AI Linguistic Analysis
        truncated_text = article_text[:2000]
        print("🧠 Analyzing linguistic patterns...")
        result = self.classifier(truncated_text)[0]

        # 2. Fact Check Cross-Reference
        fact_check_results = self.fact_check_claim(search_claim)

        return {
            "prediction": result['label'],
            "confidence": round(result['score'] * 100, 2),
            "preview_text": textwrap.shorten(article_text, width=150, placeholder="..."),
            "fact_check": fact_check_results
        }

# ==============================================================================
# TESTING THE MULTI-MODAL PIPELINE
# ==============================================================================
if __name__ == "__main__":
    engine = FakeNewsEngine()

    test_cases = [
        {
            "url": "https://en.wikipedia.org/wiki/Moon_landing_conspiracy_theories",
            "claim": "Moon landing was faked"
        }
    ]

    for test in test_cases:
        print("-" * 80)
        print(f"🌐 Target: {test['url']}")
        report = engine.analyze_article(test['url'], test['claim'])

        if "error" in report:
            print(f"❌ Error: {report['error']}")
        else:
            print("\n📊 --- FINAL MULTI-MODAL REPORT ---")
            print(f"Snippet: {report['preview_text']}\n")

            if report['prediction'] == 'FAKE':
                print(f"🤖 AI Verdict: 🔴 LIKELY FAKE/UNRELIABLE ({report['confidence']}% confidence)")
            else:
                print(f"🤖 AI Verdict: 🟢 LIKELY REAL/FACTUAL ({report['confidence']}% confidence)")

            print(f"\n📚 FACT-CHECK EVIDENCE:\n{report['fact_check']}")
        print("-" * 80 + "\n")

🤖 Initializing Fake News Transformer Engine...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded successfully!

--------------------------------------------------------------------------------
🌐 Target: https://en.wikipedia.org/wiki/Moon_landing_conspiracy_theories
🧠 Analyzing linguistic patterns...
🔎 Cross-referencing databases for: 'Moon landing was faked'...

📊 --- FINAL MULTI-MODAL REPORT ---
Snippet: Moon landing conspiracy theories Conspiracy theories claim that some or all elements of the Apollo program and the associated Moon landings were...

🤖 AI Verdict: 🟢 LIKELY REAL/FACTUAL (100.0% confidence)

📚 FACT-CHECK EVIDENCE:
- Source: Moon landing conspiracy theories | Context: Conspiracy theories claim that some or all elements of the Apollo program and the associated Moon landings were hoaxes s...
- Source: Moon landing conspiracy theories in popular culture | Context: &quot;NASA Moon Landing&quot; tested and debunked common claims made by Moon landing conspiracy theorists. On the SyFy s...
---------------------------------------------------------------------

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
import textwrap
import requests
import trafilatura
import threading
import time
from fastapi.middleware.cors import CORSMiddleware
from google.colab import output

# 1. Initialize the Web Server
app = FastAPI()

# Allow your local browser extension to talk to this cloud server securely
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)

# 2. Load the AI Model
print("🤖 Initializing AI Model (Please wait)...")
classifier = pipeline("text-classification", model="winterForestStump/Roberta-fake-news-detector")
print("✅ AI Model Loaded!")

class AnalyzeRequest(BaseModel):
    url: str = ""
    text: str = ""

def fetch_wikipedia_evidence(claim):
    """Searches Wikipedia API for historical or factual context matching the claim."""
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "opensearch",
        "search": claim,
        "limit": 2,
        "namespace": 0,
        "format": "json"
    }
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        res = requests.get(search_url, params=params, headers=headers, timeout=5).json()
        titles = res[1]
        snippets = res[2]

        evidence_list = []
        for i in range(len(titles)):
            evidence_list.append(f"Source: {titles[i]} | Context: {snippets[i]}")
        return evidence_list if evidence_list else ["No explicit fact-check matches found on Wikipedia."]
    except Exception as e:
        return [f"Fact-check search bypassed: {str(e)}"]

# 3. Define the Unified API Endpoint
@app.post("/analyze")
def analyze_article(req: AnalyzeRequest):
    content = ""
    source_used = ""

    # Strategy A: Use Direct Text from the extension screen (Bulletproof!)
    if req.text.strip() and len(req.text.strip()) > 100:
        content = req.text
        source_used = "Extension Screen Capture"
    # Strategy B: Fallback to URL Web Scraping
    elif req.url.strip():
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36"}
        try:
            response = requests.get(req.url, headers=headers, timeout=10)
            if response.status_code == 200:
                content = trafilatura.extract(response.text)
        except:
            pass
        source_used = "URL Web Scraper"

    if not content:
        # If both fail or text was too short, use a default fallback to keep it from crashing
        content = req.text if req.text.strip() else "Unknown source text."
        source_used = "Fallback Context Input"

    # 4. Process Text with AI
    result = classifier(content, truncation=True, max_length=512)[0]

    # 5. Extract a quick claim headline for the Fact-Checker
    title_claim = content.split('\n')[0][:60] if content else "News updates"
    evidence = fetch_wikipedia_evidence(title_claim)

    return {
        "status": "success",
        "source_method": source_used,
        "verdict": result["label"],
        "confidence": round(result["score"] * 100, 2),
        "snippet": textwrap.shorten(content, width=120, placeholder="..."),
        "fact_check": evidence
    }

# 6. Start the Live Server and Expose the Cloud Port
nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("\n🔌 Establishing secure Localtunnel bridge...")
import urllib

# Get Colab's public IP for the security screen
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"\n🔐 SECURITY PASSWORD (Copy this IP): {ip}\n")

# Start localtunnel
!npm install -g localtunnel > /dev/null 2>&1
!lt --port 8000

🤖 Initializing AI Model (Please wait)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ AI Model Loaded!

🔌 Establishing secure Localtunnel bridge...

🔐 SECURITY PASSWORD (Copy this IP): 34.74.145.238

your url is: https://calm-toes-cough.loca.lt
